In [ ]:
import pandas as pd
import numpy as np
import re

df_jantung = pd.read_csv('cppt_jantung_extract.csv')
df_ranap = pd.read_csv('cppt_ranap_extract.csv')
df_remed = pd.read_csv('resume_medis_extract.csv')

df = pd.concat([df_jantung, df_ranap, df_remed], ignore_index=True)

In [ ]:
def parse_age(s):
    patt = re.compile(r'(?:(\d+)\s*thn)?\s*(?:(\d+)\s*bln)?\s*(?:(\d+)\s*hr)?', re.I)
    t, b, h = patt.fullmatch(s.strip()).groups()
    return int(t or 0), int(b or 0), int(h or 0)

def parse_age_column(df):
    tahun, bulan, hari = zip(*df['pasien.umur'].map(parse_age))
    df['age_days'] = np.array(tahun) * 365.25 + np.array(bulan) * 30.44 + np.array(hari)
    return df

def remove_age_outliers(df, lower, upper):
    return df[(df['age_days'] >= lower) & (df['age_days'] <= upper)]

def get_quota(df, max_total=1000, max_per_group=50):
    df = df.copy()
    df['pasien.jeniskelamin'] = df['pasien.jeniskelamin'].map({'Perempuan': 'F', 'Laki-Laki': 'M'})
    supply = df.pivot_table(index='grouped_diagnosa', columns='pasien.jeniskelamin', aggfunc='size', fill_value=0)
    flat_supply = {(diag, gender): count for diag, row in supply.iterrows() for gender, count in row.items()}
    sorted_supply = sorted(flat_supply.items(), key=lambda x: x[1], reverse=True)

    quota = {}
    used_f = 0
    used_m = 0
    max_per_gender = max_total // 2

    for (diag, gender), count in sorted_supply:
        if gender == 'F' and used_f >= max_per_gender:
            continue
        if gender == 'M' and used_m >= max_per_gender:
            continue
        available_quota = min(count, max_per_group)
        if gender == 'F':
            take = min(available_quota, max_per_gender - used_f)
            used_f += take
        else:
            take = min(available_quota, max_per_gender - used_m)
            used_m += take
        if take > 0:
            quota[(diag, gender)] = take
        if used_f + used_m >= max_total:
            break
    return quota

def draw_quota_sample(df, quota, on=['grouped_diagnosa', 'pasien.jeniskelamin']):
    out = []
    for key, n in quota.items():
        gdf = df
        for col, val in zip(on, key):
            gdf = gdf[gdf[col] == val]
        if len(gdf) >= n:
            out.append(gdf.sample(n, random_state=0))
    if not out:
        return pd.DataFrame()
    return pd.concat(out, ignore_index=True)


In [ ]:
df2 = df.copy()
df2 = parse_age_column(df2)

df_clean = remove_age_outliers(df2, 365, 28000)

df = df_clean.copy()
df['pasien.jeniskelamin'] = df['pasien.jeniskelamin'].map({
    'Perempuan': 'F',
    'Laki-Laki': 'M'
})

supply = df.pivot_table(
    index='grouped_diagnosa',
    columns='pasien.jeniskelamin',
    aggfunc='size',
    fill_value=0
)

flat_supply = {
    (diag, gender): count
    for diag, row in supply.iterrows()
    for gender, count in row.items()
}

sorted_supply = sorted(flat_supply.items(), key=lambda x: x[1], reverse=True)

max_total = 3000
max_per_gender = max_total // 2 
max_per_group = 50              

quota = {}
used_f = 0
used_m = 0

for (diag, gender), count in sorted_supply:
    if gender == 'F' and used_f >= max_per_gender:
        continue
    if gender == 'M' and used_m >= max_per_gender:
        continue

    available_quota = min(count, max_per_group)
    if gender == 'F':
        take = min(available_quota, max_per_gender - used_f)
        used_f += take
    else:
        take = min(available_quota, max_per_gender - used_m)
        used_m += take

    if take > 0:
        quota[(diag, gender)] = take

    if used_f + used_m >= max_total:
        break


def draw_exact(df, quota, on):
    out = []
    for key, n in quota.items():
        gdf = df
        for col, val in zip(on, key):
            gdf = gdf[gdf[col] == val]
        if len(gdf) < n:
            continue  
        out.append(gdf.sample(n, random_state=0))
    return pd.concat(out, ignore_index=True)


df_final = draw_exact(df, quota, ['grouped_diagnosa', 'pasien.jeniskelamin'])

print("shape:", df_final.shape)
print("Female:", (df_final['pasien.jeniskelamin'] == 'F').sum())
print("Male:", (df_final['pasien.jeniskelamin'] == 'M').sum())
print("Diagnosa:", df_final.groupby('grouped_diagnosa').size())

tahun, bulan, hari = zip(*df_final['pasien.umur'].map(parse_age))
df_final['age_days'] = np.array(tahun)*365.25 + np.array(bulan)*30.44 + np.array(hari)
cut_tuamuda = 49.53 * 365.25            
bins   = [0, cut_tuamuda, np.inf]    
labels = ['muda', 'tua']
df_final['age_cat'] = pd.cut(df_final['age_days'], bins=bins, labels=labels, right=True)

In [ ]:
df_final = df_final.reset_index()
df_final.to_csv("3000_data_gabungan.csv")